# `latitude` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `latitude`.

            ## Relationships selected in advance

            - `longitude` — The two values are one inseparable geographic coordinate.
- `region` — Named region should broadly agree with coordinate location.
- `lga` — Administrative geography gives a categorical back-off for location.
- `gps_height` — Coordinate and height missingness overlap and both describe physical location.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'latitude'
feature_metadata = {'order': 7, 'name': 'latitude', 'audit_type': 'coordinate', 'role': 'candidate', 'disposition': 'retain only as part of a validated coordinate pair', 'finding': 'Latitude near zero participates in the paired missing-location sentinel.', 'decision': 'Create one coordinate-availability flag and transform latitude and longitude together.', 'risk': 'Using either coordinate independently breaks location meaning and encourages spatial memorisation.', 'sentinel_values': [-2e-08], 'related': [{'feature': 'longitude', 'reason': 'The two values are one inseparable geographic coordinate.'}, {'feature': 'region', 'reason': 'Named region should broadly agree with coordinate location.'}, {'feature': 'lga', 'reason': 'Administrative geography gives a categorical back-off for location.'}, {'feature': 'gps_height', 'reason': 'Coordinate and height missingness overlap and both describe physical location.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for latitude.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,longitude,The two values are one inseparable geographic ...
1,region,Named region should broadly agree with coordin...
2,lga,Administrative geography gives a categorical b...
3,gps_height,Coordinate and height missingness overlap and ...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,latitude,longitude,Spearman correlation,-0.3619,59400,57517,57516,NaN,NaN,The two values are one inseparable geographic ...
1,latitude,region,correlation ratio (eta),0.9692,59400,57517,21,NaN,NaN,Named region should broadly agree with coordin...
2,latitude,lga,correlation ratio (eta),0.9948,59400,57517,125,NaN,NaN,Administrative geography gives a categorical b...
3,latitude,gps_height,Spearman correlation,-0.0874,59400,57517,2428,NaN,NaN,Coordinate and height missingness overlap and ...


In [3]:
usable_coordinates = (
    training_features["longitude"].between(28, 42)
    & training_features["latitude"].between(-13, 0)
)
coordinate_evidence = pd.DataFrame({
    "training rows": [len(training_features)],
    "usable coordinate rows": [usable_coordinates.sum()],
    "paired sentinel/out-of-bounds rows": [(~usable_coordinates).sum()],
    "paired sentinel/out-of-bounds (%)": [(~usable_coordinates).mean() * 100],
}, index=["longitude + latitude"]).round(2)
display(coordinate_evidence)


,training rows,usable coordinate rows,paired sentinel/out-of-bounds rows,paired sentinel/out-of-bounds (%)
longitude + latitude,59400,57588,1812,3.05


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `latitude`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **retain only as part of a validated coordinate pair**.
